# V2 raw RTF to paragraph Parquet

This notebook uses the standalone V2 paragraph pipeline. It can inspect one local or GCS document and can run the resumable GCS-to-GCS batch pipeline. Both paths convert raw RTF into normalized, globally numbered paragraphs and write:

- `paragraphs.parquet` — one row per paragraph;
- `numbered_document.json` — the complete numbered text and paragraph count.

No language model or GPU is required. The later classification and extraction handlers are intentionally not run here.

## Dependencies

Use the project environment when running locally. In a clean Colab runtime, uncomment and run the installation command.

In [ ]:
# Colab only:
# %pip install -q "pyarrow>=24,<25" "striprtf>=0.0.32,<0.0.33" google-cloud-bigquery google-cloud-storage

## Import the V2 paragraph pipeline

The bootstrap supports starting Jupyter from the repository root or any directory beneath it.

In [ ]:
from pathlib import Path
import sys

import pyarrow.parquet as pq
from IPython.display import display

cwd = Path.cwd().resolve()
repository_root = None
source_root = None

for candidate in (cwd, *cwd.parents):
    candidate_source = candidate / "src"
    if (candidate_source / "document_split" / "__init__.py").exists():
        repository_root = candidate
        source_root = candidate_source
        break
    if (candidate / "document_split" / "__init__.py").exists():
        source_root = candidate
        repository_root = candidate.parent
        break

if source_root is None or repository_root is None:
    raise RuntimeError(
        "Could not locate src/document_split. Run the notebook from the "
        "cloned repository or add its src directory to sys.path."
    )

if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from document_split.v2 import (
    DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS,
    DocumentTextParsingSettings,
    V2_INFO_VERSION,
    create_google_cloud_clients,
    parse_document_to_artifacts,
    run_document_text_parsing_pipeline,
)

print(f"Repository: {repository_root}")
print(f"V2 version: {V2_INFO_VERSION}")

## Configure the source document

Use `SOURCE_MODE = "local"` for a file already on disk. Use `SOURCE_MODE = "gcs"` to download the standard `{justice_kind}/{document_id}.rtf` object from Cloud Storage.

In [ ]:
SOURCE_MODE = "local"  # "local" or "gcs"

JUSTICE_KIND = 2
DOCUMENT_ID = "117888886"

# Local source configuration
LOCAL_RTF_PATH = repository_root / f"{DOCUMENT_ID}.rtf"

# GCS authentication. In Colab, store the service-account JSON in a secret
# named cloud_access and grant this notebook access to that secret.
GCS_AUTH_MODE = "colab_secret"  # "colab_secret", "colab_user", or "adc"
COLAB_SERVICE_ACCOUNT_SECRET = "cloud_access"
GCP_PROJECT_ID = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.project_id
BIGQUERY_TABLE = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.bigquery_table

# GCS source configuration; required only when SOURCE_MODE == "gcs"
GCS_SOURCE_BUCKET = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.source_bucket
GCS_SOURCE_OBJECT = f"{JUSTICE_KIND}/{DOCUMENT_ID}.rtf"

# Batch pipeline configuration. Execution remains disabled until explicitly enabled.
RUN_GCS_PIPELINE = False
GCS_SOURCE_PREFIX = ""
GCS_DESTINATION_BUCKET = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.destination_bucket
GCS_DESTINATION_PREFIX = DEFAULT_DOCUMENT_TEXT_PARSING_SETTINGS.destination_prefix
PIPELINE_JUSTICE_KINDS = (2,)
PIPELINE_BATCH_SIZE = 500
PIPELINE_MAX_WORKERS = 5
# Smoke run: process at most 10 unparsed BigQuery rows.
# Set to None only when you are ready for the full run.
PIPELINE_LIMIT = 10
OVERWRITE_EXISTING = False
SHOW_PROGRESS = True

# Local artifacts preserve the V2 cloud-style directory structure.
OUTPUT_DIR = (
    repository_root
    / "artifacts"
    / GCS_DESTINATION_PREFIX
    / str(JUSTICE_KIND)
    / DOCUMENT_ID
)

print(f"Source mode: {SOURCE_MODE}")
print(f"Output directory: {OUTPUT_DIR}")

## Configure Google Cloud authentication

The same credentials are used for BigQuery document selection, source downloads, destination uploads, and `is_parsed` updates.

- `colab_secret` — recommended for this project; reads service-account JSON from the Colab secret configured by `COLAB_SERVICE_ACCOUNT_SECRET`.
- `colab_user` — opens the interactive Colab Google sign-in flow.
- `adc` — uses Application Default Credentials, suitable for a configured local workstation or service account environment.

The metadata-service `404` error occurs when `adc` is used in a runtime that has no attached service account.

In [ ]:
def notebook_google_cloud_clients():
    return create_google_cloud_clients(
        project_id=GCP_PROJECT_ID,
        auth_mode=GCS_AUTH_MODE,
        colab_service_account_secret=COLAB_SERVICE_ACCOUNT_SECRET,
    )

## Load the raw RTF

In [ ]:
if SOURCE_MODE == "local":
    if not LOCAL_RTF_PATH.is_file():
        raise FileNotFoundError(f"RTF file does not exist: {LOCAL_RTF_PATH}")
    raw_rtf = LOCAL_RTF_PATH.read_bytes()
elif SOURCE_MODE == "gcs":
    if not GCS_SOURCE_BUCKET:
        raise ValueError("Set GCS_SOURCE_BUCKET before using GCS mode")
    storage_client = notebook_google_cloud_clients().storage
    raw_rtf = (
        storage_client.bucket(GCS_SOURCE_BUCKET)
        .blob(GCS_SOURCE_OBJECT)
        .download_as_bytes()
    )
else:
    raise ValueError("SOURCE_MODE must be either 'local' or 'gcs'")

print(f"Loaded {len(raw_rtf):,} bytes for document {DOCUMENT_ID}")

## Run the shared RTF-to-artifacts processor

This API performs the same RTF cleanup, paragraph splitting, and global numbering used by both the automated paragraph pipeline and the complete V2 pipeline. It does not use a tokenizer or model.

In [ ]:
state = parse_document_to_artifacts(
    document_id=DOCUMENT_ID,
    justice_kind=JUSTICE_KIND,
    raw_rtf=raw_rtf,
    output_dir=OUTPUT_DIR,
)

paragraphs_path = state.artifact_path("paragraphs.parquet")
numbered_document_path = state.artifact_path("numbered_document.json")

print(f"Paragraphs: {len(state.paragraphs):,}")
print(f"Parquet: {paragraphs_path}")
print(f"Numbered JSON: {numbered_document_path}")

## Validate and inspect the Parquet output

The file intentionally has one row per paragraph. The final extraction pipeline later merges handler results into one document-level row.

In [ ]:
paragraph_table = pq.read_table(paragraphs_path, use_threads=False)

expected_columns = [
    "document_id",
    "paragraph_index",
    "paragraph_order",
    "numbered_text",
    "text",
]
assert paragraph_table.column_names == expected_columns
assert paragraph_table.num_rows == len(state.paragraphs)
assert paragraph_table.column("paragraph_index").to_pylist() == list(
    range(1, paragraph_table.num_rows + 1)
)

print(paragraph_table.schema)
display(paragraph_table.to_pandas().head(20))

## Preview the numbered document

In [ ]:
preview_characters = 5_000
print(state.numbered_text[:preview_characters])
if len(state.numbered_text) > preview_characters:
    print("\n... preview truncated ...")

## Run the automated BigQuery-driven pipeline

The notebook is configured for a 10-document smoke run with `PIPELINE_LIMIT = 10`. Set `RUN_GCS_PIPELINE = True` after configuring authentication, the `document_data` table, and buckets. The pipeline selects at most 10 rows where `is_parsed IS NOT TRUE`, builds `{source_prefix}/{justice_kind}/{doc_id}.rtf`, downloads the source RTF, and uses the same batch processing and BigQuery updates as production. After each batch, only successfully uploaded or already-complete documents are updated to `is_parsed = TRUE`; failures remain unparsed.

```text
document_text_parsing/info_version_9/{justice_kind}/{document_id}/
├── _document.json
├── paragraphs.parquet
└── numbered_document.json
```

A manifest and run logs are stored under `document_text_parsing/info_version_9/`. The Parquet object is uploaded last and acts as the completion marker. After validating the smoke result, set `PIPELINE_LIMIT = None` and rerun this cell for the full dataset.

In [ ]:
if RUN_GCS_PIPELINE:
    pipeline_settings = DocumentTextParsingSettings(
        project_id=GCP_PROJECT_ID,
        bigquery_table=BIGQUERY_TABLE,
        source_bucket=GCS_SOURCE_BUCKET,
        destination_bucket=GCS_DESTINATION_BUCKET,
        source_prefix=GCS_SOURCE_PREFIX,
        destination_prefix=GCS_DESTINATION_PREFIX,
        justice_kinds=PIPELINE_JUSTICE_KINDS,
        batch_size=PIPELINE_BATCH_SIZE,
        max_workers=PIPELINE_MAX_WORKERS,
        limit=PIPELINE_LIMIT,
        overwrite_existing=OVERWRITE_EXISTING,
        show_progress=SHOW_PROGRESS,
    )
    run_label = (
        f"smoke run (limit={PIPELINE_LIMIT})"
        if PIPELINE_LIMIT is not None
        else "full run"
    )
    print(f"Starting {run_label}")
    pipeline_clients = notebook_google_cloud_clients()
    pipeline_result = run_document_text_parsing_pipeline(
        pipeline_settings,
        storage_client=pipeline_clients.storage,
        bigquery_client=pipeline_clients.bigquery,
    )
    print(pipeline_result)
else:
    print(
        "Batch execution disabled. Set RUN_GCS_PIPELINE=True after "
        "configuring authentication and bucket settings."
    )